# 06 — v0.3 存储加固:工作区生命周期与可验证历史

本 notebook 逐节演示 **v0.3.0 存储加固战役**(Stage A)交付的功能,并标出与 v0.2 的行为差异。
只使用 factgraph 自身的公开面,不涉及任何上层应用。

## 与 v0.2 的变更总览

| 领域 | v0.2 | v0.3 |
|---|---|---|
| 持久化 | 写驻内存,`save_workspace()` 整库快照;不 save = 丢弃 | **写穿即持久**(每笔写 = 一个 SQLite 事务);`save_workspace()` 退化为 metadata 更新 |
| 并发 | 无保护(多开静默读旧/分叉) | **写者独占**(flock):双开显式失败 |
| 写身份 | 无 | 每笔提交推进 **tx 链**(`tx_seq`/`tx_id`),批量一批一 tx |
| 完整性 | 无内容校验 | **LtHash16-v2 state_digest** 绑定事实内容;打开时 fail-closed 校验;显式 `repair` |
| meta / schema | 直写,无历史 | **入 tx 历史链**(meta 不影响事实集指纹;schema 走 transition 链) |
| 写成本 | O(账本)(1M 条时 ~3.2s/笔) | **O(delta)**(~1-2ms,与账本规模无关) |
| bytes 字段 | 仅部分生命周期 | 全生命周期一致 |
| 旧工作区 | — | `python -m factgraph migrate-workspace` 一次性迁移 |

> 完整变更清单见 `CHANGELOG.md`;模块级行为契约见 `src/factgraph/core/store/docs/README.md` 与 `src/factgraph/sdk/docs/00_user_guide.en.md` §11。


In [1]:
import os, time, shutil, sqlite3, tempfile, subprocess, sys
from pathlib import Path

from factgraph.sdk import FactGraph, Database
from factgraph.sdk.schema import Entity, Field, Identity
from factgraph.sdk.compile import compile_schema_from_classes
from factgraph.core.store.database import (
    DatabaseError,
    DatabaseIntegrityError,
    DatabaseLockedError,
)

ROOT = tempfile.mkdtemp(prefix="fg_v030_demo_")
print("演示工作目录:", ROOT)


class Case(Entity):
    case_id: str = Identity()
    title: str = Field()
    amount: int = Field()
    attachment: bytes = Field()


演示工作目录: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/fg_v030_demo_9tyf3ng9


## §1 写穿即持久:不再需要 save

v0.2 的心智模型是"改动驻内存,`save_workspace()` 落盘"。v0.3 里**每一笔写在返回前就已经持久**
(数据、head、指纹状态在同一个 SQLite 事务里提交)。下面全程不调用 save,关闭后重新打开,数据都在:


In [2]:
ws1 = f"{ROOT}/case_ws"

fg = FactGraph.create(path=ws1, schema_classes=[Case])
fg.entities.create(Case, case_id="C-001")
ref = fg.entities.ref(Case, case_id="C-001")
fg.fields.set(Case.title, ref, "Missing invoice")
fg.fields.set(Case.amount, ref, 1200)
fg.close()                      # 注意:全程没有 save_workspace()

fg = FactGraph.load_workspace(ws1, schema_classes=[Case])
ref = fg.entities.ref(Case, case_id="C-001")
print("title  =", fg.fields.get(Case.title, ref))
print("amount =", fg.fields.get(Case.amount, ref))


title  = Missing invoice
amount = 1200


> **⚠️ Breaking(v0.2 → v0.3)**:"load → 改 → 不 save 当作丢弃" 的模式**不再成立** —— 写入即持久。
> 需要 dry-run 语义时,正确做法是**目录拷贝**(在副本工作区上实验)。
> `save_workspace()` 仍可调用,但只更新 workspace metadata 时间戳,是无害 no-op:


In [3]:
fg.save_workspace()      # 无害:只 touch metadata,不做任何数据写入
print("save_workspace() 返回,零数据变更")


save_workspace() 返回,零数据变更


## §2 写者独占与显式 close

v0.3 工作区持有 OS 级排他锁(flock):同一工作区第二次打开会**显式失败**(而不是 v0.2 那样静默共存、
彼此读到旧数据)。`close()` 幂等,也支持 context manager:


In [4]:
# fg 仍持有 ws1 → 第二次打开显式失败
try:
    FactGraph.load_workspace(ws1, schema_classes=[Case])
except Exception as e:
    print("双开被拒 →", type(e).__name__, "|", str(e)[:80])

fg.close()
fg.close()               # 幂等,双 close 无害

# close 后的写入给出 SDK 层错误(带错误码),不是裸的底层异常
try:
    fg.fields.set(Case.title, ref, "after close")
except Exception as e:
    print("close 后写入 →", type(e).__name__, "|", str(e)[:80])

# context manager:退出自动释放锁
with FactGraph.load_workspace(ws1, schema_classes=[Case]) as fg_cm:
    r = fg_cm.entities.ref(Case, case_id="C-001")
    print("with 块内可读:", fg_cm.fields.get(Case.title, r))
print("with 块退出,锁已释放")


双开被拒 → SDKStoreError | Database workspace is already open for writing: /var/folders/05/6btr2vg13b9gvgs3
close 后写入 → SDKStoreError | fg.fields.set is unavailable because this FactGraph's Database is closed; create
with 块内可读: Missing invoice
with 块退出,锁已释放


## §3 tx 链:每笔写都有身份

每次提交推进一条哈希链:`tx_seq`(提交序)、`tx_id`(承诺**本笔全部有序变更**的链头)。
`Database` 是公开面 —— 用 `FactGraph.attach(db)` 车道可以直接观察链的推进。
注意**批量语义:一个 batch = 一个 tx**(里面不管多少操作)。


In [5]:
schema_ir = compile_schema_from_classes([Case])
db = Database.create(path=f"{ROOT}/chain_ws", schema_ir=schema_ir)
fg3 = FactGraph.attach(db, schema_classes=[Case])

print("初始 tx_seq:", db.head().tx_seq)
fg3.entities.create(Case, case_id="C-100")
print("entities.create      → tx_seq", db.head().tx_seq)
r100 = fg3.entities.ref(Case, case_id="C-100")
fg3.fields.set(Case.title, r100, "batch demo")
print("fields.set           → tx_seq", db.head().tx_seq)

with fg3.batch() as tx:                     # 3 个操作,1 个 tx
    c = tx.entity(Case, case_id="C-101")
    c.title.set("created in batch")
    c.amount.set(7)
    tx.commit(objects=[c])
print("3 操作的 batch       → tx_seq", db.head().tx_seq, " (一批 = 一 tx)")

h = db.head()
print()
print("tx_id        =", h.tx_id[:40], "...")
print("state_digest =", h.state_digest[:40], "...")
print("schema_digest=", h.schema_digest[:40], "...")


初始 tx_seq: 0
entities.create      → tx_seq 1
fields.set           → tx_seq 2
3 操作的 batch       → tx_seq 3  (一批 = 一 tx)

tx_id        = tx:98decb5f2d12ac90a2160a7b23dd33216ac7e ...
state_digest = lthash16-v2:xe1mgacCT8tpL4RIXuOFJIxTwIR_ ...
schema_digest= sha256:9353b57146775f0c8cabab52793804705 ...


## §3b 双承诺:状态指纹 vs 历史链

v0.3 把"指纹"拆成两个各司其职的承诺:

- **`state_digest`** — 当前**活跃事实集**的纯函数(LtHash 多重集哈希,增量维护 O(delta)):
  回答 *"两个副本的当前事实是否一致"*;
- **`tx_id` 链** — 承诺**到达此状态的完整有序历史**:两条不同的历史即使收敛到同一事实集,
  链头也不同。

下面从同一基础工作区复制出两份,各自走一段**不同的临时历史**(写入不同值再撤销),
终态事实集相同 → `state_digest` 相等,`tx_id` 不同:


In [6]:
fg3.close()
db.close()

for name in ("hist_a", "hist_b"):
    shutil.copytree(f"{ROOT}/chain_ws", f"{ROOT}/{name}")

def transient_round(path, value):
    d = Database.open(path=path, schema_ir=schema_ir)
    g = FactGraph.attach(d, schema_classes=[Case])
    rr = g.entities.ref(Case, case_id="C-100")
    aid = g.fields.set(Case.amount, rr, value)   # 写入一个临时值…
    g.assertions.retract(aid)                    # …随即撤销:终态回到基础事实集
    g.close()
    head = d.head()
    d.close()
    return head

ha = transient_round(f"{ROOT}/hist_a", 111)
hb = transient_round(f"{ROOT}/hist_b", 222)

print("两侧 state_digest 相等:", ha.state_digest == hb.state_digest, " (同一事实集)")
print("两侧 tx_id 不同:      ", ha.tx_id != hb.tx_id, " (不同历史)")


两侧 state_digest 相等: True  (同一事实集)
两侧 tx_id 不同:       True  (不同历史)


## §4 防篡改:内容绑定 + fail-closed + repair 的边界

指纹绑定每条断言的**内容**(不只是 id)。这带来两条不同强度的保证,分别演示:

1. **内容伪造不可洗白**:绕过 API 改事实内容 → 打开 fail-closed,而且 **`repair` 同样拒绝**
   (每条断言携带内容绑定的 `assertion_digest`,被改过的行自证矛盾 —— 伪造无法经 repair 变成新事实,
   正确恢复路径是备份/副本);
2. **记账漂移可显式修复**:如果坏的是**状态记录**(数据本身可信,只是指纹账目损坏),
   `repair` 以账本数据为真相源重算,并把带 reason 的 repair 事件**追加进历史链**(审计可见)。


In [7]:
# —— 保证 1:内容伪造,open 与 repair 双双拒绝 ——
tamper_ws = f"{ROOT}/hist_a"
con = sqlite3.connect(f"{tamper_ws}/db/assertions.db")
n = con.execute(
    "UPDATE claims SET rest_terms = replace(rest_terms, 'batch demo', 'TAMPERED')"
).rowcount
con.commit(); con.close()
print(f"绕过 API 篡改了 {n} 行事实内容")

try:
    Database.open(path=tamper_ws, schema_ir=schema_ir)
except DatabaseIntegrityError as e:
    print("打开被拒  →", str(e)[:80], "...")
try:
    Database.repair(tamper_ws, schema_ir=schema_ir, reason="attempt to launder tamper")
except DatabaseIntegrityError as e:
    print("repair 拒绝 →", str(e)[:80], "...")
print("→ 伪造内容无法洗白;该副本的正确恢复路径 = 备份/未污染副本")
print()

# —— 保证 2:记账漂移(数据可信,状态记录损坏),repair 显式修复 ——
drift_ws = f"{ROOT}/hist_b"
con = sqlite3.connect(f"{drift_ws}/db/assertions.db")
con.execute("UPDATE ledger_meta SET value = 'lthash16-v2:AAAA' WHERE key = 'head_state_digest'")
con.commit(); con.close()
print("破坏了状态记录(head_state_digest),数据行未动")

try:
    Database.open(path=drift_ws, schema_ir=schema_ir)
except DatabaseIntegrityError as e:
    print("打开被拒  →", str(e)[:80], "...")

repaired = Database.repair(drift_ws, schema_ir=schema_ir,
                           reason="notebook demo: rebuild state records from trusted ledger data")
print("repair 成功 → tx_seq", repaired.head().tx_seq, "(repair 事件带 reason 入链)")
repaired.close()


绕过 API 篡改了 7 行事实内容
打开被拒  → assertion_digest does not match factual content for asrt:e6a6c0e6dc81477094b89f5 ...
repair 拒绝 → assertion_digest does not match factual content for asrt:e6a6c0e6dc81477094b89f5 ...
→ 伪造内容无法洗白;该副本的正确恢复路径 = 备份/未污染副本

破坏了状态记录(head_state_digest),数据行未动
打开被拒  → invalid head_state_digest: LtHash state must be exactly 2048 bytes, got 3 ...
repair 成功 → tx_seq 6 (repair 事件带 reason 入链)


> 诚实边界:能同时改写数据行、每条 assertion_digest meta 与全部状态记录的全能攻击者仍可伪造一个
> 自洽的假账本 —— 对抗这一级需要外部锚点(签名/时间戳),已登记为后续设计入口。
> 本节展示的保证是:**任何未把全套账目一起改掉的篡改,必然在打开时暴露,且无法经 repair 洗白**。

## §5 meta 与 schema 也进历史 —— 双承诺的直观展示

`append_meta` 会推进历史链(它是一个事件),但**不改变 `state_digest`**(meta 不属于事实集承诺)。
schema 变更走 **transition 链**(承诺 old→new digest),同样不动事实集指纹:


In [8]:
d = Database.open(path=f"{ROOT}/hist_b", schema_ir=schema_ir)
g = FactGraph.attach(d, schema_classes=[Case])
rr = g.entities.ref(Case, case_id="C-100")
aid = g.fields.set(Case.title, rr, "meta demo")

before = d.head()
g.assertions.append_meta(aid, "provenance_class", "verified_by_ops")
after = d.head()
print("append_meta:  tx_seq", before.tx_seq, "→", after.tx_seq, " (meta 事件入链)")
print("              state_digest 不变:", before.state_digest == after.state_digest)

# Database 保留的身份键不可经公开面伪造
try:
    g.assertions.append_meta(aid, "assertion_digest", "sha256:" + "0" * 64)
except Exception as e:
    print("保留键被拒 →", type(e).__name__)

class Attachment(Entity):
    attachment_id: str = Identity()
    label: str = Field()

g.add_schema_classes(Attachment)
h2 = d.head()
print("add_schema_classes: tx_seq →", h2.tx_seq, " (schema transition 入链)")
print("              schema_digest 变化:", h2.schema_digest != after.schema_digest)
print("              state_digest 仍不变:", h2.state_digest == after.state_digest)

g.close(); d.close()


append_meta:  tx_seq 7 → 8  (meta 事件入链)
              state_digest 不变: True
保留键被拒 → SDKStoreError
add_schema_classes: tx_seq → 9  (schema transition 入链)
              schema_digest 变化: True
              state_digest 仍不变: True


## §6 bytes 字段:全生命周期一致

`bytes` 是账本的一等值类型;v0.3 起在所有生命周期(create/load/attach/batch)行为一致:


In [9]:
with FactGraph.create(path=f"{ROOT}/bytes_ws", schema_classes=[Case]) as g2:
    g2.entities.create(Case, case_id="C-B")
    rb = g2.entities.ref(Case, case_id="C-B")
    g2.fields.set(Case.attachment, rb, b"\x89PNG\r\n...binary-payload")
    val = g2.fields.get(Case.attachment, rb)
    print(type(val).__name__, "→", val[:12], "...")


bytes → b'\x89PNG\r\n...bin' ...


## §7 v0.2 工作区迁移

旧工作区(v0.2 布局)在 v0.3 下**拒绝写模式打开**,并指向一次性迁移 CLI。
迁移流程:staging 构建 → 打开验证 → 原子替换 → 旧工作区归档为可见的 `<name>.legacy-<时间戳>` 副本。
支持 `--dry-run` 预览:


In [10]:
repo = Path.cwd() if (Path.cwd() / "src" / "factgraph").exists() else Path.cwd().parent
env = dict(os.environ, PYTHONPATH=str(repo / "src"))
out = subprocess.run([sys.executable, "-m", "factgraph", "migrate-workspace", "--help"],
                     capture_output=True, text=True, env=env, cwd=repo)
print(out.stdout or out.stderr)


usage: python -m factgraph migrate-workspace [-h] [--dry-run]
                                             [--archive | --no-archive]
                                             path

positional arguments:
  path          Closed v0.2 workspace directory to migrate.

options:
  -h, --help    show this help message and exit
  --dry-run     Validate the source and print the plan without writing.
  --archive     (Default) Retain the complete v0.2 workspace plus any legacy
                registry marker.
  --no-archive  Discard the v0.2 backup after the verified replacement
                succeeds.



## §8 写性能:与账本规模脱钩

v0.2 每笔写要对**全账本**重算指纹(实测 100K 条时 ~303ms,1M 条时 ~3.2s);
v0.3 的指纹增量维护,写成本 **O(delta)**。权威证据是战役基准(`tools/benchmarks/`):
**1.05ms@100K vs 1.59ms@1M** —— 账本大 10 倍,单笔写几乎不变。

下面 300 笔连续写入只为直观感受量级(微型样本存在毫秒级抖动,不构成曲线证据):


In [11]:
with FactGraph.create(path=f"{ROOT}/perf_ws", schema_classes=[Case]) as g3:
    g3.entities.create(Case, case_id="P-1")
    rp = g3.entities.ref(Case, case_id="P-1")
    times = []
    for i in range(300):
        t0 = time.perf_counter()
        g3.fields.set(Case.title, rp, f"note-{i}")
        times.append((time.perf_counter() - t0) * 1000)

print(f"300 笔全部落盘;单笔中位 {sorted(times)[150]:.2f}ms,最大 {max(times):.2f}ms —— 全程毫秒量级")
print("(对照 v0.2:账本到 10 万条时每笔已需 ~300ms)")


300 笔全部落盘;单笔中位 2.33ms,最大 5.52ms —— 全程毫秒量级
(对照 v0.2:账本到 10 万条时每笔已需 ~300ms)


## §9 小结

本次战役交付的心智模型三句话:

1. **写 = 持久 + 有身份**:每笔写是一个原子事务,推进一条可重放、可审计的历史链;
2. **状态与历史分开承诺**:`state_digest` 答"现在是什么",`tx_id` 链答"怎么来的" ——
   meta/schema 变更进历史但不搅动事实集指纹;
3. **完整性是默认姿态**:打开必校验(fail-closed),恢复必显式(repair 带审计),并发必独占(flock)。

| 想深入 | 去处 |
|---|---|
| 全部行为变更(Breaking 清单) | `CHANGELOG.md` |
| commit 协议 / 双承诺 / 锁语义 | `src/factgraph/core/store/docs/README.md` |
| 生命周期用户指南 | `src/factgraph/sdk/docs/00_user_guide.en.md` §11 |
| 迁移 CLI 细节 | `python -m factgraph migrate-workspace --help` |

> 下一战役(Slice 3b)预告:7 表 → 3 表、claim_meta 事件化、meta 分级(结构/辖域/语义/纯迹)——
> 本 notebook 展示的 tx 链正是它们的地基。
